# 第三篇配套 Demo：从用户原话到结构化任务

这个 notebook 对应《上下文工程系列教程（三）：从用户原话到结构化任务，Agent 如何组织高信噪比上下文》。

文章里的核心链路是：

```text
用户原话 -> 任务信号提取 -> 结构化输入 -> 模型/Agent 执行 -> 结构化输出 -> 后续流程
```

本 demo 用一个真实业务场景来跑通这条链路：**电商售后客服工单 Agent**。

这个场景里，用户输入经常是口语化、带情绪、信息不完整的；但系统后续需要做工单路由、澄清提问、状态更新、自动处理建议。这正好适合演示：如何从松散原话里提取最小的高信噪比任务集合。

## 1. 依赖安装

这个 notebook 的主体 demo 不依赖真实 LLM API，主要用 `pydantic` 定义结构化输入输出，用 `rich` 更清楚地展示结果。

`openai` 是可选依赖：最后一节会给出如何把规则提取器替换成 LLM 提取器的接口位置。

In [ ]:
%pip install -q pydantic rich openai

In [ ]:
import json
import os
import re
from typing import Any, Literal, Optional

from pydantic import BaseModel, Field, ValidationError
from rich import print
from rich.panel import Panel
from rich.table import Table

## 2. 业务场景：售后客服工单 Agent

假设我们要做一个售后客服 Agent。用户可能直接输入一段自然语言，例如：

> 昨天买的耳机左耳没声音，订单 A20240618，别给我打电话，直接告诉我能不能换，最好今天处理。

如果直接把这句话塞给 Agent，模型需要自己判断：

- 用户真正目标是什么？退款、换货、维修，还是投诉？
- 用户提供了哪些材料？订单号、商品、故障描述、时间信息分别是什么？
- 哪些是硬约束？例如“别打电话”。
- 哪些是偏好？例如“最好今天处理”。
- 当前缺什么信息？例如收货状态、故障凭证、是否超过售后期。
- 后续应该直接处理、澄清，还是转人工？

下面的代码会把这些判断显式化。

In [ ]:
raw_inputs = [
    "昨天买的耳机左耳没声音，订单 A20240618，别给我打电话，直接告诉我能不能换，最好今天处理。",
    "我这个包裹一直没到，物流显示三天没动了，订单号 B99881，能不能退款？越快越好。",
    "你们这个杯子有划痕，包装也破了。我不想来回折腾，看看怎么处理吧。",
    "订单 D77777，耳机右耳坏了，已上传故障视频，申请换货。",
    "不是退款，我是想换一个同款新的；地址别用老地址，我晚点发新地址。",
]

for i, text in enumerate(raw_inputs, start=1):
    print(Panel(text, title=f"用户原话 {i}", expand=False))

## 3. 定义结构化输入 Schema

文章里提到，任务信号提取至少要关注五类信息：目标、材料、约束、偏好和验收标准。

在工程里，我们还会额外保留三类信息：

- `missing_information`：还缺什么，决定是否需要澄清。
- `assumptions`：模型或程序推断了什么，避免把推断误当成事实。
- `evidence`：每个关键信号来自用户明说、系统推断，还是当前缺失。

In [ ]:
SignalSource = Literal["explicit", "inferred", "missing"]
RiskLevel = Literal["low", "medium", "high"]
NextStep = Literal["answer", "clarify", "route_to_agent", "execute_workflow"]


class Evidence(BaseModel):
    field: str
    value: str
    source: SignalSource
    note: str


class StructuredTaskInput(BaseModel):
    raw_user_input: str
    goal: str
    materials: dict[str, str] = Field(default_factory=dict)
    constraints: list[str] = Field(default_factory=list)
    preferences: list[str] = Field(default_factory=list)
    acceptance_criteria: list[str] = Field(default_factory=list)
    missing_information: list[str] = Field(default_factory=list)
    assumptions: list[str] = Field(default_factory=list)
    risk_level: RiskLevel = "medium"
    next_step: NextStep = "clarify"
    confidence: float = Field(ge=0, le=1)
    evidence: list[Evidence] = Field(default_factory=list)


def show_json(data: BaseModel | dict[str, Any]):
    if isinstance(data, BaseModel):
        data = data.model_dump()
    print(json.dumps(data, ensure_ascii=False, indent=2))

## 4. 任务信号提取：把文章里的要点变成代码

真实系统里，任务信号提取可以由规则、表单、分类器或 LLM 完成。这里先用规则做一个可运行版本，重点不是规则本身多聪明，而是观察它如何把文章里的抽象原则落到字段上。

这个提取器会做几件事：

- 从用户原话中识别目标。
- 提取可用材料，例如订单号、商品、问题描述、时间要求。
- 区分硬约束和偏好。
- 显式标出缺失信息。
- 区分用户明确表达和系统推断。
- 根据风险和缺失信息决定下一步。

In [ ]:
ORDER_PATTERN = re.compile(r"(?:订单号?|订单)[：:\s]*([A-Z]\d{4,})", re.I)


def add_evidence(items: list[Evidence], field: str, value: str, source: SignalSource, note: str):
    items.append(Evidence(field=field, value=value, source=source, note=note))


def infer_goal(text: str, evidence: list[Evidence]) -> tuple[str, list[str]]:
    assumptions = []
    if "退款" in text:
        add_evidence(evidence, "goal", "申请退款", "explicit", "用户明确提到退款")
        return "申请退款", assumptions
    if "换" in text or "换货" in text or "同款新的" in text:
        add_evidence(evidence, "goal", "申请换货", "explicit", "用户明确表达换货诉求")
        return "申请换货", assumptions
    if "没到" in text or "物流" in text:
        add_evidence(evidence, "goal", "处理物流异常", "inferred", "用户描述包裹未到或物流停滞")
        assumptions.append("把物流异常作为当前主任务")
        return "处理物流异常", assumptions
    if "划痕" in text or "破" in text or "没声音" in text or "坏" in text:
        add_evidence(evidence, "goal", "处理商品质量售后", "inferred", "用户描述商品质量或破损问题")
        assumptions.append("用户可能需要换货、退款或维修，但尚未明确处理方式")
        return "处理商品质量售后", assumptions

    add_evidence(evidence, "goal", "识别售后诉求", "missing", "用户没有给出清晰售后目标")
    return "识别售后诉求", ["售后类型暂不明确"]


def extract_materials(text: str, evidence: list[Evidence]) -> dict[str, str]:
    materials = {}

    order_match = ORDER_PATTERN.search(text)
    if order_match:
        materials["order_id"] = order_match.group(1)
        add_evidence(evidence, "materials.order_id", materials["order_id"], "explicit", "从用户原话中提取订单号")

    product_keywords = ["耳机", "杯子", "包裹"]
    for keyword in product_keywords:
        if keyword in text:
            materials["product_or_item"] = keyword
            add_evidence(evidence, "materials.product_or_item", keyword, "explicit", "用户提到具体商品或包裹")
            break

    issue_keywords = ["没声音", "没到", "三天没动", "划痕", "包装也破了", "破了", "坏了"]
    found_issues = [keyword for keyword in issue_keywords if keyword in text]
    if found_issues:
        materials["issue_description"] = "；".join(found_issues)
        add_evidence(evidence, "materials.issue_description", materials["issue_description"], "explicit", "用户描述了问题现象")

    proof_keywords = ["照片", "视频", "截图", "已上传"]
    found_proof = [keyword for keyword in proof_keywords if keyword in text]
    if found_proof:
        materials["proof"] = "；".join(found_proof)
        add_evidence(evidence, "materials.proof", materials["proof"], "explicit", "用户提到已提供或可提供凭证")

    if "昨天" in text:
        materials["purchase_time_hint"] = "昨天"
        add_evidence(evidence, "materials.purchase_time_hint", "昨天", "explicit", "用户提供了时间线索")

    return materials


def extract_constraints_and_preferences(text: str, evidence: list[Evidence]) -> tuple[list[str], list[str]]:
    constraints = []
    preferences = []

    if "别给我打电话" in text or "不要打电话" in text:
        constraints.append("不要电话联系用户")
        add_evidence(evidence, "constraints", "不要电话联系用户", "explicit", "这是联系方式限制，属于硬约束")

    if "地址别用老地址" in text:
        constraints.append("不要使用旧地址")
        add_evidence(evidence, "constraints", "不要使用旧地址", "explicit", "地址使用会影响履约，属于硬约束")

    if "最好今天" in text or "越快越好" in text or "尽快" in text:
        preferences.append("优先快速处理")
        add_evidence(evidence, "preferences", "优先快速处理", "explicit", "用户表达了时效偏好")

    if "不想来回折腾" in text:
        preferences.append("减少用户往返沟通和操作成本")
        add_evidence(evidence, "preferences", "减少用户往返沟通和操作成本", "explicit", "用户表达了体验偏好")

    if "直接告诉我" in text:
        preferences.append("直接给出可执行结论")
        add_evidence(evidence, "preferences", "直接给出可执行结论", "explicit", "用户不希望收到泛泛解释")

    return constraints, preferences


def build_acceptance_criteria(goal: str) -> list[str]:
    base = ["说明当前判断依据", "给出下一步处理动作"]
    if goal in {"申请退款", "申请换货"}:
        return ["确认订单是否满足售后条件", "说明是否需要补充凭证", "给出处理结果或澄清问题"]
    if goal == "处理物流异常":
        return ["确认物流异常状态", "判断是否可退款或催派", "给出下一步处理建议"]
    return base


def find_missing_information(goal: str, materials: dict[str, str], constraints: list[str], evidence: list[Evidence]) -> list[str]:
    missing = []
    if "order_id" not in materials:
        missing.append("订单号")
        add_evidence(evidence, "missing_information", "订单号", "missing", "没有订单号就无法查询售后状态")
    if "issue_description" not in materials:
        missing.append("具体问题描述")
        add_evidence(evidence, "missing_information", "具体问题描述", "missing", "缺少问题现象会影响分类和处理")
    if goal in {"申请退款", "申请换货", "处理商品质量售后"} and "proof" not in materials:
        missing.append("商品状态或问题凭证")
        add_evidence(evidence, "missing_information", "商品状态或问题凭证", "missing", "售后处理通常需要凭证或状态确认")
    if "不要使用旧地址" in constraints:
        missing.append("新收货地址")
        add_evidence(evidence, "missing_information", "新收货地址", "missing", "用户要求不要用旧地址，但尚未给出新地址")
    return missing


def decide_risk_and_next_step(goal: str, missing: list[str]) -> tuple[RiskLevel, NextStep, float]:
    risk: RiskLevel = "medium"
    if goal in {"申请退款", "申请换货"}:
        risk = "high"
    elif goal == "处理物流异常":
        risk = "medium"

    critical_missing = {"订单号", "新收货地址"}.intersection(missing)
    if critical_missing:
        return risk, "clarify", 0.65
    if missing and risk == "high":
        return risk, "clarify", 0.72
    if risk == "high":
        return risk, "route_to_agent", 0.82
    return risk, "execute_workflow", 0.86


def extract_task_signals(raw_user_input: str) -> StructuredTaskInput:
    evidence: list[Evidence] = []
    goal, assumptions = infer_goal(raw_user_input, evidence)
    materials = extract_materials(raw_user_input, evidence)
    constraints, preferences = extract_constraints_and_preferences(raw_user_input, evidence)
    acceptance_criteria = build_acceptance_criteria(goal)
    missing = find_missing_information(goal, materials, constraints, evidence)
    risk, next_step, confidence = decide_risk_and_next_step(goal, missing)

    return StructuredTaskInput(
        raw_user_input=raw_user_input,
        goal=goal,
        materials=materials,
        constraints=constraints,
        preferences=preferences,
        acceptance_criteria=acceptance_criteria,
        missing_information=missing,
        assumptions=assumptions,
        risk_level=risk,
        next_step=next_step,
        confidence=confidence,
        evidence=evidence,
    )

In [ ]:
tasks = [extract_task_signals(text) for text in raw_inputs]

for i, task in enumerate(tasks, start=1):
    print(Panel(json.dumps(task.model_dump(), ensure_ascii=False, indent=2), title=f"结构化任务输入 {i}", expand=False))

### 观察：不要把推断当成用户明确表达

任务信号提取最容易出错的地方，是把系统推断出来的内容当成用户明确说过的话。

所以下面把每个字段的来源展开看：`explicit` 是用户明说，`inferred` 是系统推断，`missing` 是当前仍然缺失。

In [ ]:
def show_evidence_table(task: StructuredTaskInput):
    table = Table(title="任务信号来源")
    table.add_column("字段")
    table.add_column("值")
    table.add_column("来源")
    table.add_column("说明")

    for item in task.evidence:
        table.add_row(item.field, item.value, item.source, item.note)
    print(table)


show_evidence_table(tasks[0])

## 5. 结构化输入：把任务信号压缩成执行器真正需要的上下文

结构化输入不是为了“看起来整齐”，而是为了让后续流程更稳定。

下面把一段杂乱的多轮对话压缩成执行器需要的最小任务上下文。注意：结构化对象不一定比单句原话更短，但它能显著减少无关背景，并把关键字段放到稳定位置。

In [ ]:
messy_history = """
用户：我之前买过好几次你们家的东西，整体还行。
用户：但是这次有点烦，昨天买的耳机左耳没声音。
用户：订单 A20240618。
用户：别给我打电话，我白天开会不方便。
用户：你们别发那种很长的模板，直接告诉我能不能换。
用户：最好今天处理，我明天出差。
客服：请问是否可以提供故障视频？
""".strip()

compact_task = extract_task_signals(messy_history)

executor_context = {
    "goal": compact_task.goal,
    "materials": compact_task.materials,
    "constraints": compact_task.constraints,
    "preferences": compact_task.preferences,
    "acceptance_criteria": compact_task.acceptance_criteria,
    "missing_information": compact_task.missing_information,
    "next_step": compact_task.next_step,
}

print(Panel(messy_history, title="原始多轮上下文", expand=False))
print(Panel(json.dumps(executor_context, ensure_ascii=False, indent=2), title="给执行器的结构化输入", expand=False))
print({
    "raw_chars": len(messy_history),
    "structured_chars": len(json.dumps(executor_context, ensure_ascii=False)),
    "note": "这里比较的是字符数。真实系统里应按 tokenizer 统计 token。",
})

## 6. 澄清与任务确认：不要把误解格式化

文章里强调：任务信号提取不是一次完美识别，而是一套“提取、校验、澄清、执行、纠偏”的闭环。

如果关键信息缺失，Agent 不应该强行执行，而应该把缺失信息变成可回答的问题。

In [ ]:
def build_confirmation(task: StructuredTaskInput) -> str:
    lines = [
        f"我的理解是：你希望我处理「{task.goal}」。",
        f"我已识别到的材料：{task.materials or '暂无明确材料'}。",
    ]
    if task.constraints:
        lines.append(f"需要遵守的限制：{'；'.join(task.constraints)}。")
    if task.preferences:
        lines.append(f"你的偏好：{'；'.join(task.preferences)}。")
    if task.missing_information:
        lines.append(f"当前还缺：{'；'.join(task.missing_information)}。")
    return "\n".join(lines)


def build_clarifying_questions(task: StructuredTaskInput) -> list[str]:
    question_map = {
        "订单号": "请补充订单号，方便查询售后状态。",
        "具体问题描述": "请简单说明遇到的具体问题或异常现象。",
        "商品状态或问题凭证": "请提供商品当前状态或问题凭证，例如照片、视频或物流截图。",
        "新收货地址": "请提供新的收货地址，避免后续换货发到旧地址。",
    }
    return [question_map[item] for item in task.missing_information if item in question_map]


needs_help = tasks[2]
print(Panel(build_confirmation(needs_help), title="任务确认", expand=False))
print(Panel("\n".join(build_clarifying_questions(needs_help)), title="澄清问题", expand=False))

## 7. 用户反馈是纠偏信号，不是失败终点

当用户说“不是这个意思”时，系统应该更新任务对象，而不是只把它当成一次失败。

下面演示：第一次提取把诉求理解成退款；用户反馈后，任务对象被修正为换货，并补充新的缺失信息。

In [ ]:
def apply_user_feedback(previous_task: StructuredTaskInput, feedback: str) -> StructuredTaskInput:
    merged_raw_input = previous_task.raw_user_input + "\n用户反馈：" + feedback
    updated = extract_task_signals(merged_raw_input)

    # 保留上一轮已经明确的材料，除非新输入覆盖它。
    merged_materials = dict(previous_task.materials)
    merged_materials.update(updated.materials)

    return updated.model_copy(update={"materials": merged_materials})


first_task = extract_task_signals("订单 C12345，杯子破了，能不能退款？")
feedback = "不是退款，我是想换一个同款新的；地址别用老地址，我晚点发新地址。"
updated_task = apply_user_feedback(first_task, feedback)

print(Panel(json.dumps(first_task.model_dump(), ensure_ascii=False, indent=2), title="反馈前任务对象", expand=False))
print(Panel(json.dumps(updated_task.model_dump(), ensure_ascii=False, indent=2), title="反馈后任务对象", expand=False))

## 8. 结构化输出：让模型结果可以被系统继续消费

结构化输出的目标不是让答案变得机械，而是让系统能稳定判断下一步：

- 是否需要继续澄清？
- 是否可以进入自动工作流？
- 是否需要转人工？
- 哪些状态需要写回工单？

下面定义一个客服 Agent 的输出结构。

In [ ]:
OutputStatus = Literal["need_clarification", "ready_to_process", "handoff", "completed"]


class ActionItem(BaseModel):
    name: str
    arguments: dict[str, Any] = Field(default_factory=dict)
    requires_human_approval: bool = False


class StructuredAgentOutput(BaseModel):
    status: OutputStatus
    user_message: str
    questions: list[str] = Field(default_factory=list)
    actions: list[ActionItem] = Field(default_factory=list)
    state_updates: dict[str, Any] = Field(default_factory=dict)
    confidence: float = Field(ge=0, le=1)
    validation_notes: list[str] = Field(default_factory=list)


def generate_agent_output(task: StructuredTaskInput) -> StructuredAgentOutput:
    if task.next_step == "clarify":
        questions = build_clarifying_questions(task)
        return StructuredAgentOutput(
            status="need_clarification",
            user_message="我还需要补充少量信息，才能继续处理。",
            questions=questions,
            actions=[],
            state_updates={"ticket_goal": task.goal, "blocked_by": task.missing_information},
            confidence=task.confidence,
            validation_notes=["关键信息缺失，先澄清比直接执行更稳妥"],
        )

    actions = [
        ActionItem(name="query_order", arguments={"order_id": task.materials.get("order_id")}),
        ActionItem(name="check_after_sales_policy", arguments={"goal": task.goal}),
    ]
    if task.goal == "申请换货":
        actions.append(ActionItem(name="create_exchange_draft", arguments={"order_id": task.materials.get("order_id")}, requires_human_approval=True))
    elif task.goal == "申请退款":
        actions.append(ActionItem(name="create_refund_draft", arguments={"order_id": task.materials.get("order_id")}, requires_human_approval=True))

    return StructuredAgentOutput(
        status="ready_to_process",
        user_message="我已整理好处理信息，可以先查询订单和售后规则，再生成处理草案。",
        questions=[],
        actions=actions,
        state_updates={
            "ticket_goal": task.goal,
            "risk_level": task.risk_level,
            "contact_constraints": task.constraints,
        },
        confidence=task.confidence,
        validation_notes=["结构化输入已通过字段校验", "高风险动作需要人工确认后执行"],
    )

In [ ]:
for i, task in enumerate(tasks, start=1):
    output = generate_agent_output(task)
    print(Panel(json.dumps(output.model_dump(), ensure_ascii=False, indent=2), title=f"结构化输出 {i}", expand=False))

## 9. 校验与失败处理：结构化不是只看起来像 JSON

如果模型输出的 JSON 无法解析，字段缺失，类型错误，或者语义和 schema 不匹配，系统必须有处理策略。

下面用一个故意错误的输出演示校验失败。

In [ ]:
bad_output = {
    "status": "maybe_done",  # 不在枚举范围内
    "user_message": "应该可以处理",
    "confidence": 1.4,  # 超出 0 到 1
}

try:
    StructuredAgentOutput.model_validate(bad_output)
except ValidationError as exc:
    print(Panel(str(exc), title="校验失败示例", expand=False))

## 10. 可选扩展：把规则提取器替换成 LLM 提取器

上面的规则提取器适合教学，因为它完全可运行、可观察。但在真实开放式输入里，通常可以把 `extract_task_signals()` 替换成一次 LLM 调用。

关键点不是“必须用 LLM”，而是保持同一个结构化契约：无论信号来自规则、表单、分类器还是模型，后续执行器看到的都应该是同一种 `StructuredTaskInput`。

In [ ]:
USE_LLM = False


def extract_task_signals_with_llm(raw_user_input: str) -> StructuredTaskInput:
    """示例接口：真实项目中可以在这里调用 LLM，并把返回 JSON 校验成 StructuredTaskInput。"""
    if not USE_LLM:
        return extract_task_signals(raw_user_input)

    # 伪代码：
    # 1. 把 StructuredTaskInput 的字段说明放进 system / developer prompt。
    # 2. 要求模型只返回 JSON。
    # 3. 用 StructuredTaskInput.model_validate_json(...) 做解析和校验。
    # 4. 校验失败时重试、降级到澄清问题，或转人工。
    raise NotImplementedError("请在这里接入你的模型调用。")


demo_task = extract_task_signals_with_llm(raw_inputs[0])
show_json(demo_task)

## 11. 小结

这个 notebook 对应第三篇文章的三个核心要点：

1. **用户输入处理**：从用户原话中提取目标、材料、约束、偏好和验收标准，同时显式表达缺失信息和不确定性。
2. **结构化输入**：把任务信号组织成后续流程能消费的任务对象，减少模型自行猜重点的负担。
3. **结构化输出**：让 Agent 的结果可以被解析、校验、路由和写回状态，而不是只生成一段自由文本。

回到本系列的核心原则：上下文工程不是追求把所有信息都塞进去，而是找到最小的高信噪比 token 集合，让模型和系统更稳定地产生期望结果。